In [1]:
library(Seurat)
library(Signac)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86)
library(ggplot2)
library(patchwork)
library(hdf5r) 
library(future)
library(RColorBrewer)
library(dplyr)
library(Matrix)
library(BSgenome.Hsapiens.UCSC.hg38)
library(glue)
library(harmony)
library(matrixStats)
library(scales)
library(biomaRt)
library(curl)
library(goseq)
library(httr)
library(Scillus)
library(TFBSTools)
library(JASPAR2020)
library(ggridges)
library(ggrepel)
library(ggsignif)
library(qusage)
library(tidyverse)
httr::set_config(config(ssl_verifypeer = 0L))
set.seed(1234)
setwd("/home/jupyter/scATAC_analysis/edit/snatac-rcc-manuscript")
source("scripts/functions.r")

Attaching SeuratObject

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: ensembldb

Loading required packag

# Table S2

## Sheet A: Differentially accessible genes between each RCC tumor cell cluster and all other RCC tumor clusters

In [2]:
differential_geneactivity_alltum = readRDS('processed_data/RCC_tumor_DAGs.rds')

# filter to significant and save
write.table(differential_geneactivity_alltum %>% filter(p_val_adj < 0.05), file = 'tables/s2a_RCCtumorstates_diffgenes.txt', sep = '\t', quote = F, col.names = T, row.names = F)

## Sheet B: RCC tumor state assignments per cell barcode

In [3]:
tumor <- readRDS("processed_data/rcc_tumor_cells_seurat.rds")


In [5]:
rcc_tumor_state_annotations = tumor@meta.data %>% filter(annotation != 'Excluded') %>% dplyr::select(annotation)
rcc_tumor_state_annotations$`Cell barcode` = row.names(rcc_tumor_state_annotations)
rcc_tumor_state_annotations$`RCC tumor state` = rcc_tumor_state_annotations$annotation
rcc_tumor_state_annotations = rcc_tumor_state_annotations %>% dplyr::select(`Cell barcode`,  `RCC tumor state`)
write.table(rcc_tumor_state_annotations, file = "tables/s2b_RCC_tumor_state_cell_barcode.txt", sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
head(rcc_tumor_state_annotations)


,Cell barcode,RCC tumor state
,<chr>,<chr>
C3L-00004-01_GGGCCATGTTCTACCC-1,C3L-00004-01_GGGCCATGTTCTACCC-1,ccRCC balanced
C3L-00004-01_TCACCACGTACAAGCG-1,C3L-00004-01_TCACCACGTACAAGCG-1,ccRCC balanced
C3L-00004-01_CCCTCTCGTAAAGGCC-1,C3L-00004-01_CCCTCTCGTAAAGGCC-1,ccRCC balanced
C3L-00004-01_TAGGAGGCAATGTAAG-1,C3L-00004-01_TAGGAGGCAATGTAAG-1,ccRCC balanced
C3L-00004-01_GCATTGACACCTGTGG-1,C3L-00004-01_GCATTGACACCTGTGG-1,ccRCC balanced
C3L-00004-01_TTGCTTAAGGCGATTG-1,C3L-00004-01_TTGCTTAAGGCGATTG-1,ccRCC balanced


## Sheet C: Differentially accessible peaks between each ccRCC tumor state and all other ccRCC tumor states

In [6]:
da_peaks = readRDS('processed_data/ccRCC_balaned_DAPs.rds')
# Filter to significant
sigpeaks = da_peaks %>% 
    filter(p_val_adj < 0.05) %>% 
    rename(peak = gene)
sigpeaks$row.names = NULL
# Set names as used in figures
sigpeaks$cluster = paste0('C',sigpeaks$cluster)

write.table(sigpeaks, 'tables/s2c_ccrcc_states_diffpeaks.txt', quote = F, row.names = F, col.names = T, sep = '\t')

## Sheet D: ccRCC tumor state assignments per barcode and C1-C3 signature scores

Read in ccRCC balanced seurat object

In [15]:
ccrcc <- readRDS("processed_data/ccrcc_balanced_seurat.rds")
ccrcc@meta.data$`ccRCC tumor state` = paste0('C', ccrcc@meta.data$seurat_clusters)

Calculate signature scores

In [16]:
results <- list()
for (cluster in c("C1", "C2", "C3")) {
    peaks <- (sigpeaks %>% filter(cluster == !!cluster))$peak
    sig <- calculate_signature_scores(seurat_object = ccrcc, sig_peak_set = peaks, pct.open = 0.1, final_score_name = glue("{cluster} program score"))
    results[[cluster]] <- sig
}
signature_df <- dplyr::bind_cols(results)
ccrcc <- AddMetaData(ccrcc, signature_df)


Matching GC.percent distribution

Matching GC.percent distribution

Matching GC.percent distribution

Warning message:
"Invalid name supplied, making object name syntactically valid. New object name is C1.program.scoreC2.program.scoreC3.program.score; see ?make.names for more details on syntax validity"


Save data

In [17]:
ccrcc_states_sigs = ccrcc@meta.data %>% dplyr::select(`ccRCC tumor state`, C1.program.score, C2.program.score, C3.program.score)
ccrcc_states_sigs = ccrcc_states_sigs %>% rename(
    `C1 program score`         = C1.program.score,
    `C2 program score`         = C2.program.score,
    `C3 program score`         = C3.program.score
)
ccrcc_states_sigs$`Cell barcode` = row.names(ccrcc_states_sigs)
ccrcc_states_sigs = ccrcc_states_sigs %>% dplyr::select(`Cell barcode`,  `ccRCC tumor state`, `C1 program score`, `C2 program score`, `C3 program score`)
write.table(ccrcc_states_sigs, file = "tables/s2d_ccRCC_tumor_state_sigs_cell_barcode.txt", sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
head(ccrcc_states_sigs)


,Cell barcode,ccRCC tumor state,C1 program score,C2 program score,C3 program score
,<chr>,<chr>,<dbl>,<dbl>,<dbl>
C3L-00004-01_GGGCCATGTTCTACCC-1,C3L-00004-01_GGGCCATGTTCTACCC-1,C3,-0.17128826,-0.14232126,0.19105322
C3L-00004-01_TCACCACGTACAAGCG-1,C3L-00004-01_TCACCACGTACAAGCG-1,C1,-0.16775423,-0.24049060,-0.03407486
C3L-00004-01_CCCTCTCGTAAAGGCC-1,C3L-00004-01_CCCTCTCGTAAAGGCC-1,C0,-0.24190581,-0.15478981,-0.10694435
C3L-00004-01_TAGGAGGCAATGTAAG-1,C3L-00004-01_TAGGAGGCAATGTAAG-1,C3,-0.28274175,-0.17346450,0.14914626
C3L-00004-01_GCATTGACACCTGTGG-1,C3L-00004-01_GCATTGACACCTGTGG-1,C1,0.08657560,-0.26369836,-0.24872973
C3L-00004-01_TTGCTTAAGGCGATTG-1,C3L-00004-01_TTGCTTAAGGCGATTG-1,C2,0.05331048,-0.04197311,-0.20245948


## Sheet E: GREAT pathway enrichment results for ccRCC epigenetic program peak sets

In [7]:
results = list()

Cluster 1

In [8]:
great_results <- readRDS('processed_data/C1_GREAT_pathway_results.rds')
dim(great_results)
results[['C1']] = great_results

[1] 14 23

Cluster 2

In [9]:
great_results <- readRDS('processed_data/C2_GREAT_pathway_results.rds')
dim(great_results)
results[['C2']] = great_results

[1] 518  23

Cluster 3

In [10]:
great_results <- readRDS('processed_data/C3_GREAT_pathway_results.rds')
dim(great_results)
results[['C3']] = great_results

[1] 199  23

In [11]:
write.table(dplyr::bind_rows(results), 'tables/s2e_ccrcc_states_GREAT_output.txt', sep = '\t', quote = F, row.names = F, col.names = T)

## Sheet F: Transcription factor binding site motif enrichment results for ccRCC epigenetic program peak sets

In [12]:
enriched.motifs.all = readRDS("processed_data/ccRCC_balaned_enriched_motifs.rds")

In [13]:
write.table(enriched.motifs.all %>% filter(p.adjust < 0.05),
           file = 'tables/s2f_ccrcc_states_tfmotifs.txt', sep = '\t', quote = F, row.names = F, col.names = T)

## Sheet G: Associations between epigenetic modifier mutations and ccRCC tumor program scores

In [14]:
summary_df = readRDS('processed_data/epimuts_vs_ccrccstates.rds')


colnames(summary_df) = c('Gene', 'ccRCC tumor state', 'Cliffs Delta', 'p-value', 'q-value')


write.table(summary_df,
           file = glue('tables/s2g_epimuts_vs_ccrccstates.txt'), sep = '\t', quote = F, row.names = F, col.names = T)


# Export to Excel
Switch to python kernel. 

In [1]:
import pandas as pd
import os
os.chdir('/home/jupyter/scATAC_analysis/edit/snatac-rcc-manuscript')

In [2]:
rcc_dags = pd.read_csv('tables/s2a_RCCtumorstates_diffgenes.txt', sep = '\t')
rcc_dags.head()

rcc_states = pd.read_csv('tables/s2b_RCC_tumor_state_cell_barcode.txt', sep = '\t')
rcc_states.head()

ccrcc_daps = pd.read_csv('tables/s2c_ccrcc_states_diffpeaks.txt', sep = '\t')
ccrcc_daps.head()

ccrcc_states = pd.read_csv('tables/s2d_ccRCC_tumor_state_sigs_cell_barcode.txt', sep = '\t')
ccrcc_states.head()

ccrcc_pathways = pd.read_csv('tables/s2e_ccrcc_states_GREAT_output.txt', sep ='\t')
ccrcc_pathways.head()

ccrcc_tf_motifs = pd.read_csv('tables/s2f_ccrcc_states_tfmotifs.txt', sep = '\t')
ccrcc_tf_motifs.head()

mut_progs = pd.read_csv('tables/s2g_epimuts_vs_ccrccstates.txt', sep = '\t')
mut_progs.head()

,Gene,ccRCC tumor state,Cliffs Delta,p-value,q-value
0,BAP1,C1,0.142857,0.562301,0.636528
1,BAP1,C2,0.577640,0.012693,0.076158
2,BAP1,C3,0.770186,0.000446,0.005355
3,SETD2,C1,-0.120988,0.583484,0.636528
4,SETD2,C2,0.323457,0.133107,0.532427


In [3]:
with pd.ExcelWriter('tables/table_S2_draft.xlsx') as writer:  
    rcc_dags.to_excel(writer, sheet_name='A', index = False)
    rcc_states.to_excel(writer, sheet_name='B', index = False)
    ccrcc_daps.to_excel(writer, sheet_name='C', index = False)
    ccrcc_states.to_excel(writer, sheet_name='D', index = False)
    ccrcc_pathways.to_excel(writer, sheet_name='E', index = False)
    ccrcc_tf_motifs.to_excel(writer, sheet_name='F', index = False)
    mut_progs.to_excel(writer, sheet_name='G', index = False)

Add README with titles in excel app. 